# AETHER — Stage 3 Latency Sweep and Scenario Diversity

Один Qwen backbone, несколько сценариев (успешный tool call, неудачный tool call, реплика без tool call) на развёртке MCP latency 3000/1500/750/300 мс. Каждый run проверяется явными pass/fail критериями (см. `evaluate_run` в `aether/experiments/colab_stage3.py`), а не только логируется.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
MODEL_ID = "Qwen/Qwen3-1.7B"  # @param {type:"string"}
TOOL_LATENCIES_MS = "3000,1500,750,300"  # @param {type:"string"}

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Укажи настоящий REPO_URL")

In [ ]:
import os, subprocess, sys
from pathlib import Path

subprocess.run(["nvidia-smi"], check=False)
repo_dir = Path("/content/aether")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo_dir}[dev,ml]"], check=True)
print("Commit:")
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

In [ ]:
artifacts = repo_dir / "artifacts" / "colab-stage3"
artifacts.mkdir(parents=True, exist_ok=True)
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
(artifacts / "tests.log").write_text(tests.stdout, encoding="utf-8")
print(tests.stdout)
if tests.returncode != 0:
    raise RuntimeError("Tests failed")

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(repo_dir / "src")
command = [
    sys.executable, "-m", "aether.experiments.colab_stage3",
    "--allow-download",
    "--model", MODEL_ID,
    "--tool-latency-ms", TOOL_LATENCIES_MS,
    "--output-dir", str(artifacts),
]
run = subprocess.run(command, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(artifacts / "model_run.log").write_text(run.stdout, encoding="utf-8")
print(run.stdout)
print("Exit code:", run.returncode)

In [ ]:
import json

report_path = artifacts / "report.json"
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("Status:", report.get("status"))
    print("Summary:", json.dumps(report.get("summary", {}), indent=2))
    for run in report.get("runs", []):
        print(
            run.get("scenario"),
            run.get("tool_latency_ms"),
            "PASSED" if run.get("passed") else "FAILED",
            run.get("speaker_first_token_minus_tool_complete_ms"),
            run.get("checks"),
        )

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/aether-colab-stage3-logs", "zip", root_dir=artifacts)
print(archive)
files.download(archive)